# Seeing Machines — Amin Rasti Pour

**Corpus:** personal CG / abstract works (stored on Google Drive)



### Corpus statement

**What the collection is:**  346 images drawn from my TouchDesigner practices, frames captured
across several distinct generative systems: a particle simulation, a 2D fluid/physics sim, a
solver-based system, and noise-driven / procedural-shape generators. Frames include different
camera angles, parameter states, and timeline positions within each project, rather than one
still per project.

**Why I chose it:** These are systems I built and understand at the level of their underlying
logic, not just their appearance, I know exactly what's generating each frame, which lets me evaluate whether the machine's reading of an image
matches what's structurally happening in it. My specific research interest is pareidolia:
whether a vision-language model imposes real-world objects or scenes onto visuals that are
procedurally generated and have no representational intent.

**How I curated it:** Frames were sampled across numerous TouchDesigner projects/systems, exporting
stills at different timeline states and camera positions per system to capture both
within-system variation and between-system diversity, rather than hand-picking a best of set.

## 1. Install dependencies



In [ ]:
import os
import sys
from packaging import version

def setup_environment():
    reinstall_needed = False

    try:
        import transformers
        import bitsandbytes
        import numpy as np
        # Gemma 3 requires bitsandbytes >= 0.46.1 for 4-bit stability
        if version.parse(bitsandbytes.__version__) < version.parse("0.46.1"):
            reinstall_needed = True
        # Check for the numpy 2.0 vs 1.26 conflict known to break Colab VLMs
        if version.parse(np.__version__) >= version.parse("2.0.0"):
            reinstall_needed = True
    except ImportError:
        reinstall_needed = True

    if reinstall_needed:
        print("⚙️ Configuring stable environment")
        # Force versions that are known-stable on T4 Runtimes
        !pip install -q -U "numpy<2.0.0" "bitsandbytes>=0.46.1" "transformers>=4.50.0" "accelerate"
        !pip install -q -U "gradio" "open_clip_torch" "sentence-transformers" "einops" "safetensors"
        print("\n✅ Environment prepared. Please Restart Session once (Runtime > Restart session) to apply changes permanently.")
    else:
        print("✅ Environment is stable and ready.")

setup_environment()

## 2. Imports & global config

**Embedding model: SigLIP 2.**  SigLIP 2 (Feb 2025) is a drop-in upgrade over SigLIP 1 through the exact same `AutoModel`/`AutoProcessor` interface, so nothing else in the pipeline changes. It outperforms SigLIP 1 at every model scale on zero-shot classification and image-text retrieval, which is directly the operation Level 1 depends on. One implementation detail that matters for correctness: SigLIP/SigLIP2 text encoding expects `padding="max_length", max_length=64` — the processor call below is set up for that already. `google/siglip2-base-patch16-224` is the default (86M params, fine on CPU or T4); `siglip2-so400m-patch14-384` is a stronger but heavier option.

In [ ]:
import os
import json
import time
from pathlib import Path
from datetime import datetime

import torch
import numpy as np
from PIL import Image

# ---------------------------------------------------------------------------
# GLOBAL CONFIG — edit here, nowhere else
# ---------------------------------------------------------------------------
CFG = {
    # --- Corpus & storage (Drive paths) ---
    "drive_root": "/content/drive/MyDrive/Corpus",   # Updated to your provided folder name
    "corpus_subdir": "",                                        # Set to empty if images are directly in root
    "cache_subdir": "cache",                                  # embeddings + caption JSON get written here

    # --- Level 1: joint embedding model ---
    "embedding_model": "siglip2",
    "siglip2_ckpt": "google/siglip2-base-patch16-224",
    "siglip_ckpt": "google/siglip-base-patch16-224",
    "clip_ckpt": "openai/clip-vit-base-patch32",
    "text_max_length": 64,

    # --- Level 2: captioning / generation VLM ---
    "vlm_ckpt": "google/gemma-3-4b-it",
    "vlm_4bit": True,

    # --- Retrieval ---
    "seed": 42,
    "top_k_default": 8,
    "embed_batch_size": 16,
    "image_extensions": (".jpg", ".jpeg", ".png", ".webp"),
}

DRIVE_ROOT = Path(CFG["drive_root"])
CORPUS_DIR = DRIVE_ROOT / CFG["corpus_subdir"]
CACHE_DIR = DRIVE_ROOT / CFG["cache_subdir"]

torch.manual_seed(CFG["seed"])
print("Config updated with new path. Embedding model:", CFG["embedding_model"])

## 3. Runtime check


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Active device:", device)

if device == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    print("GPU:", gpu_name)
    if "T4" in gpu_name:
        print("T4 detected -> use 4-bit loading for the VLM later (CFG['vlm_4bit'] = True), avoid bf16.")
    print("If you're just writing/debugging code, switch back to a CPU runtime to preserve GPU quota.")
else:
    print("No GPU attached — fine for Level 1 end to end at this corpus size.")


## 4. Mount Google Drive & Confirm the corpus






In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ensure cache directory exists within your project folder
CACHE_DIR.mkdir(parents=True, exist_ok=True)

if not CORPUS_DIR.exists():
    print(f"Warning: {CORPUS_DIR} not found. Creating it now.")
    CORPUS_DIR.mkdir(parents=True, exist_ok=True)

image_paths = sorted([
    p for p in CORPUS_DIR.rglob("*")
    if p.suffix.lower() in CFG["image_extensions"]
])

print(f"Corpus dir: {CORPUS_DIR}")
print(f"Found {len(image_paths)} images.")

if len(image_paths) > 0:
    print(f"Success: Found {len(image_paths)} images to process.")
else:
    print("No images found. Please ensure your images are inside the folder: ", CORPUS_DIR)

In [ ]:
# Quick visual sanity check, random selection of images, just to confirm paths & loading work
import matplotlib.pyplot as plt
import random

preview_n = min(6, len(image_paths))
# Pick random samples instead of first n
random_samples = random.sample(image_paths, preview_n)

fig, axes = plt.subplots(1, preview_n, figsize=(3 * preview_n, 3))
if preview_n == 1:
    axes = [axes]
for ax, p in zip(axes, random_samples):
    img = Image.open(p).convert("RGB")
    ax.imshow(img)
    ax.set_title(p.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import os

HF_CACHE_ROOT = DRIVE_ROOT / "hf_cache"
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE_ROOT / "hub")

print("HF cache now persisted at:", HF_CACHE_ROOT)

---
# 5. Level 1 — The Finder

### 5.1 Load the joint embedding model

`CFG["embedding_model"]` picks the checkpoint; everything below only calls `get_image_features` / `get_text_features`, so switching models later (siglip2 → siglip → clip) doesn't require touching the retrieval code.


In [ ]:
from transformers import AutoProcessor, AutoModel
import torch

def load_embedding_model(cfg, device):
    print(f"Loading Level 1 Finder ({cfg['embedding_model']})...")
    if cfg["embedding_model"] == "siglip2":
        ckpt = cfg["siglip2_ckpt"]
    elif cfg["embedding_model"] == "siglip":
        ckpt = cfg["siglip_ckpt"]
    else:
        raise ValueError(f"Unknown embedding_model: {cfg['embedding_model']}")

    processor = AutoProcessor.from_pretrained(ckpt)
    model = AutoModel.from_pretrained(ckpt).to(device).eval()
    return model, processor

# Global instances for Level 1
if 'embed_model' not in globals():
    embed_model, embed_processor = load_embedding_model(CFG, device)
    print("✅ Model loaded into memory.")
else:
    print(f"ℹ️  Model ({CFG['embedding_model']}) is already loaded and ready.")

### 5.2 Embedding the corpus once, cached to Drive

Runs once per corpus version. If cached embeddings already exist on Drive (matching the current file listing), this cell loads them instead of re-embedding, safe to re-run without wasting compute. If add/remove images later, delete the two cache files listed below (or bump a version suffix) to force a fresh pass.


In [ ]:
from tqdm.auto import tqdm
import torch
import numpy as np
from PIL import Image
import json
from datetime import datetime

EMB_PATH = CACHE_DIR / f"embeddings_{CFG['embedding_model']}.npy"
INDEX_PATH = CACHE_DIR / f"embeddings_{CFG['embedding_model']}_index.json"

def embed_images(paths, model, processor, device, batch_size):
    all_embeds = []
    for i in tqdm(range(0, len(paths), batch_size), desc="Embedding corpus"):
        batch_paths = paths[i:i + batch_size]
        images = []
        for p in batch_paths:
            try:
                images.append(Image.open(p).convert("RGB"))
            except Exception as e:
                print(f"Skipping corrupt image {p}: {e}")

        if not images: continue

        inputs = processor(images=images, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model.get_image_features(**inputs)
            feats = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs

        feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
        all_embeds.append(feats.cpu().numpy())
    return np.concatenate(all_embeds, axis=0)

current_listing = [str(p) for p in image_paths]

# --- CACHE CHECK ---
needs_computation = True
if EMB_PATH.exists() and INDEX_PATH.exists():
    try:
        with open(INDEX_PATH, "r") as f:
            cached_meta = json.load(f)

        # If paths haven't changed and model is the same, load from Drive
        if cached_meta.get("paths") == current_listing and cached_meta.get("model") == CFG["embedding_model"]:
            print("Matching embeddings found on Drive. Loading cache...")
            image_embeddings = np.load(EMB_PATH)
            needs_computation = False
    except Exception as e:
        print(f"Cache exists but failed to load ({e}). Re-computing...")

if needs_computation:
    print(f"No valid cache found. Processing {len(image_paths)} images using {device.upper()}...")
    image_embeddings = embed_images(image_paths, embed_model, embed_processor, device, CFG["embed_batch_size"])

    # Save to Drive for next time
    np.save(EMB_PATH, image_embeddings)
    with open(INDEX_PATH, "w") as f:
        json.dump({"paths": current_listing, "model": CFG["embedding_model"],
                   "created": datetime.now().isoformat()}, f, indent=2)
    print(f"Successfully embedded and cached {len(image_paths)} images to Drive.")

print("image_embeddings shape:", image_embeddings.shape)

### 5.2.1 Embedding Validation Suite
Run this to verify the integrity of the cached data loaded from Drive.

In [ ]:
import random

def run_integrity_check(embeddings, paths, model, processor, device):
    print(f"--- Integrity Check for {len(paths)} Images ---")

    # 1. Shape & Count Check
    if len(embeddings) == len(paths):
        print(f"✅ Count Match: {len(embeddings)} vectors found for {len(paths)} paths.")
    else:
        print(f"❌ Count Mismatch: {len(embeddings)} vectors vs {len(paths)} paths!")

    # 2. Statistical Health (NaNs and Normalization)
    if np.isnan(embeddings).any():
        print("❌ Health Check: Found NaN values in embeddings!")
    else:
        norms = np.linalg.norm(embeddings, axis=1)
        if np.allclose(norms, 1.0, atol=1e-3):
            print("✅ Health Check: All vectors are valid and L2-normalized.")
        else:
            print("⚠️ Health Check: Vectors are not perfectly normalized.")

    # 3. Identity (Self-Retrieval) Test
    # Pick a random index to verify mapping
    idx = random.randint(0, len(paths) - 1)
    test_path = paths[idx]

    # Generate a fresh embedding for this specific image
    img = Image.open(test_path).convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.get_image_features(**inputs)
        fresh_feat = (outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs).cpu().numpy()
    fresh_feat = fresh_feat / np.linalg.norm(fresh_feat)

    # Compare fresh embedding with the one in the cache
    cached_feat = embeddings[idx]
    similarity = np.dot(fresh_feat, cached_feat.T).item()

    if similarity > 0.99:
        print(f"✅ Identity Test: Verified mapping for '{test_path.name}' (Sim: {similarity:.4f})")
    else:
        print(f"❌ Identity Test FAILED: Mapping shift detected for '{test_path.name}' (Sim: {similarity:.4f})")

run_integrity_check(image_embeddings, image_paths, embed_model, embed_processor, device)

### 5.3 Text-query retrieval function

Embeds a query string into the same space and ranks the corpus by cosine similarity (a plain dot product, since both sides are L2-normalized).

In [ ]:
import matplotlib.pyplot as plt

def embed_text_query(query, model, processor, device, max_length):
    inputs = processor(
        text=[query], padding="max_length", max_length=max_length, return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        outputs = model.get_text_features(**inputs)
        feats = outputs.pooler_output if hasattr(outputs, "pooler_output") else outputs
    feats = feats / feats.norm(p=2, dim=-1, keepdim=True)
    return feats.cpu().numpy()[0]

def search(query, top_k=None, min_similarity=None):
    top_k = top_k or CFG["top_k_default"]
    q_emb = embed_text_query(query, embed_model, embed_processor, device, CFG["text_max_length"])
    sims = image_embeddings @ q_emb
    order = np.argsort(-sims)[:top_k]
    results = [(image_paths[i], float(sims[i])) for i in order]
    if min_similarity is not None:
        results = [r for r in results if r[1] >= min_similarity]
    return results

def show_results(query, top_k=None, min_similarity=None):
    results = search(query, top_k=top_k, min_similarity=min_similarity)
    n = len(results)
    if n == 0:
        print(f'No results for "{query}" above the similarity threshold.')
        return results
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, (p, sim) in zip(axes, results):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(f"{p.name}\nsim={sim:.3f}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f'Query: "{query}"')
    plt.tight_layout()
    plt.show()
    return results

# Quick smoke test
_ = show_results("tree", top_k=5)

### 5.4 Retrieval atlas — logging queries as you go



In [ ]:
ATLAS_PATH = CACHE_DIR / "retrieval_atlas.json"

def _load_atlas():
    if ATLAS_PATH.exists():
        with open(ATLAS_PATH) as f:
            return json.load(f)
    return []

def log_atlas_entry(query, results, explanation, verdict="success"):
    atlas = _load_atlas()
    entry_idx = len(atlas) + 1

    # No longer saving thumbnails, just logging metadata
    atlas.append({
        "query": query,
        "verdict": verdict,
        "top_results": [{"file": Path(p).name, "similarity": round(sim, 4)}
                         for p, sim in results],
        "explanation": explanation,
        "logged_at": datetime.now().isoformat(),
    })

    with open(ATLAS_PATH, "w") as f:
        json.dump(atlas, f, indent=2)
    print(f"Logged #{entry_idx} -> \"{query}\" ({verdict})")
    return atlas

In [ ]:
def export_atlas_markdown(path=None):
    atlas = _load_atlas()
    if not atlas:
        print("No atlas entries logged yet.")
        return
    path = path or (CACHE_DIR / "retrieval_atlas.md")
    lines = ["# Retrieval Atlas — Level 1 (Finder)\n"]
    for i, entry in enumerate(atlas, 1):
        lines.append(f"## {i}. \"{entry['query']}\" — {entry['verdict']}\n")
        # Modified: Removed image tag reference to 'thumb' as it's no longer saved
        for r in entry["top_results"]:
            lines.append(f"- **{r['file']}** `sim={r['similarity']}`")
        lines.append(f"\n**Analysis:** {entry['explanation']}\n")
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print(f"Exported {len(atlas)} entries -> {path}")

In [ ]:
queries_to_log = [
    "bicycle", "sunset over water", "human face", "a swarm of insects",
    "flowing hair", "storm clouds", "a galaxy or nebula", "fire and smoke",
    "cracked stone or ceramic", "circuit board pattern", "dense forest canopy",
    "frog", "ripples on water", "DNA or cellular structure",
    "fabric or woven texture", "lightning", "a school of fish", "melting ice",
    "a spider web", "sparse minimal composition",
]

In [ ]:
for q in queries_to_log:
    print(f"=== {q} ===")
    show_results(q, top_k=8)

In [ ]:
atlas_notes = {
    "bicycle": ("failure", "no real bicycle match obviously, but interesting that the circular mandala/kaleidoscope patterns showed up, guess the spoke-like radial lines got associated with wheels somehow"),
    "sunset over water": ("success", "first few results are a genuinely great match, warm gradient over water reflection look, but it falls apart toward the end with random blue grid and pixel mosaic stuff"),
    "a human face": ("success", "this one's actually kind of unsettling, almost every result is resembling human face but they are affected heavily with effects ofcourse"),
    "a swarm of insects": ("mixed", "the pale scattered dots one looks like an actual swarm, but a lot of the rest is just grainy stuff that got pulled in probably cause they resemble swarms from far distance?!"),
    "flowing hair": ("mixed", "the swirly ones (pink, teal, purple) genuinely have that flowing strand look, it is kind of right on the flow part but for the hair it is hard for it to come up with anything better"),
    "storm clouds": ("mixed", "the streaky gray ones are believable as heavy cloud texture but then colorful splash show up which don't fit at all"),
    "a galaxy or nebula": ("success", "genuinely the best result set, the dark blue starfield one and the orange nebula-blob one look like space photos"),
    "fire and smoke": ("success", "the orange/black swirl and the marble sphere both have kinda the fire look, and the grainy dark one actually reads more like smoke than fire"),
    "cracked stone or ceramic": ("success", "good matches, the dark cratered textures really do look like cracked surfaces"),
    "circuit board pattern": ("success", "by far the most consistent result set out of everything, same grid screenshots keep showing up here and honestly this is the strongest, most obvious match in the whole atlas"),
    "dense forest canopy": ("mixed", "the green/purple ones from above kind of read as canopy seen from a drone, but half the results have similarities but I couldnt connect them before this, more color-matching than concept-matching I think"),
    "frog": ("failure", "kind of a joke query but wanted to see, none of these look like a frog, closest is the bumpy grainy texture which could pass as frog skin if I squint but that's a stretch"),
    "ripples on water": ("failure", "the yellow/black contour map (JFA) is a good match for ripples, rest are not good enough to pass i would say"),
    "DNA or cellular structure": ("success", "first 3 results are basically the same faint grid image repeated which isn't DNA-like at all but i seee the medical resembles for all the returned images, but the colorful blob one after that genuinely does look cell-like"),
    "fabric or woven texture": ("success", "good one, the dark woven textures look like actual fabric weave and the smooth cream swirl ones look like folded cloth, solid match across the board"),
    "lightning": ("success", "surprised me, the jagged blue streaks and the orange starburst both kinda read as electricity/sparks, this one just works"),
    "a school of fish": ("mixed", "the blue streaky line ones do look like fish swimming in formation which is cool, but then random unrelated pink and blue block stuff snuck in too"),
    "melting ice": ("mixed", "the grainy black and white textures kind of pass as ice/frost but that one colorful marble sphere at the end makes no sense here, feels like a color-based fluke"),
    "a spider web": ("success", "actually pretty good, a bunch of the green branching ones genuinely look like web strands radiating out, didn't expect that to work this well"),
    "sparse minimal composition": ("failure", "mostly pulled the same grid/graph-paper screenshots over and over, that's not sparse at all, it's dense and repetitive, feels like it just latched onto thin dark lines on light bg"),
    }

for q, (verdict, explanation) in atlas_notes.items():
    results = search(q, top_k=5)
    log_atlas_entry(q, results, explanation, verdict)

export_atlas_markdown()

### 5.5 Gradio search interface

Thin wrapper over `search()` — functional, no more.

In [ ]:
import gradio as gr

def gradio_search_fn(query, top_k):
    if not query.strip():
        return []
    results = search(query, top_k=int(top_k))
    return [(str(p), f"{p.name} | sim={sim:.3f}") for p, sim in results]

with gr.Blocks(title="Seeing Machines — Level 1: The Finder") as demo:
    gr.Markdown("## Seeing Machines — Level 1: The Finder\nSemantic search over your corpus using SigLIP 2 joint embeddings.")
    with gr.Row():
        query_box = gr.Textbox(label="Query", placeholder="e.g. 'glowing spheres', 'cracked surface texture'...", scale=3)
        topk_slider = gr.Slider(1, 20, value=CFG["top_k_default"], step=1, label="Top k", scale=1)
    search_btn = gr.Button("Search")
    gallery = gr.Gallery(label="Results", columns=4, height="auto")
    search_btn.click(fn=gradio_search_fn, inputs=[query_box, topk_slider], outputs=gallery)
    query_box.submit(fn=gradio_search_fn, inputs=[query_box, topk_slider], outputs=gallery)

demo.launch(debug=False)


# 6. Level 2 — The Companion

### 6.1 Load the VLM (Gemma 3 4B, 4-bit)
4-bit is mandatory on a T4 (no bf16 support, fp16-only compute dtype for the quant config).

In [ ]:
if 'vlm_model' in globals():
    print(f"ℹ️ VLM '{CFG['vlm_ckpt']}' is already loaded and ready.")
else:
    import torch
    from transformers import AutoProcessor, Gemma3ForConditionalGeneration, BitsAndBytesConfig

    print(f"⚙️ Loading Companion ({CFG['vlm_ckpt']})...")
    try:
        # 4-bit config is mandatory for Gemma-3-4B on a T4 GPU
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )

        vlm_processor = AutoProcessor.from_pretrained(CFG["vlm_ckpt"])
        vlm_model = Gemma3ForConditionalGeneration.from_pretrained(
            CFG["vlm_ckpt"],
            quantization_config=bnb_config,
            device_map="auto",
        )
        vlm_model.eval()
        print("✅ VLM loaded successfully.")
    except Exception as e:
        print(f"❌ Error loading VLM: {e}")
        print("\nTIP: If you just ran the setup cell for the first time, ensure you have used 'Runtime > Restart session' before running this.")

### 6.2 Description schema & captioning prompt

**Justification:** The schema splits description into a purely visual layer
(`literal_description`) and an interpretive layer (`perceived_subject`,
`resemblance_confidence`), rather than one blended `subject` field. This split exists because my
research question is specifically about pareidolia, whether the model reads real-world meaning
into procedurally generated, non-representational visuals. A single blended field would hide
that behavior; forcing the model to first describe purely in visual terms, then separately
commit to (or explicitly decline) a real-world resemblance with a stated confidence level, makes
the phenomenon I'm studying directly measurable and sortable in the caption data.

In [ ]:
CAPTION_SCHEMA_FIELDS = [
    "literal_description",   # purely visual, non-interpretive: shapes, values, motion cues — no naming of "things"
    "perceived_subject",     # what the model thinks this depicts/resembles, if anything — forces it to commit
    "resemblance_confidence",# "abstract" | "low" | "medium" | "high" — how sure it is that perceived_subject is really there
    "composition",           # framing, layout, focal point
    "color_palette",         # list of 2-4 dominant colors/tones
    "texture",               # surface qualities: smooth, grainy, glitchy, organic, etc.
    "mood",                  # emotional/atmospheric read
    "notable_elements",      # list of specific details worth indexing on
    "ambiguous_or_uncertain" # anything the model flags as unsure — separate from resemblance_confidence
]

CAPTION_PROMPT = """Describe this image for a searchable archive. This image may be an abstract generative visual (particles, simulations, noise, procedural shapes) that sometimes visually resembles real-world objects or scenes by coincidence, or may not resemble anything recognizable at all.

Respond with ONLY a JSON object, no other text, with exactly these fields:
{
  "literal_description": "purely visual terms only — shapes, values, density, motion cues. Do NOT name real-world objects here, even if tempted.",
  "perceived_subject": "if this image reminds you of or resembles a real-world object, scene, or figure, name it specifically. If it looks purely abstract with no such resemblance, say 'abstract, no resemblance'.",
  "resemblance_confidence": "one of: abstract / low / medium / high — how confident are you that perceived_subject is genuinely depicted vs. a coincidental visual similarity",
  "composition": "...",
  "color_palette": ["...", "..."],
  "texture": "...",
  "mood": "...",
  "notable_elements": ["...", "..."],
  "ambiguous_or_uncertain": "..."
}
Be specific and concrete. Do not force a resemblance that isn't there — 'abstract, no resemblance' is a valid and expected answer for most images."""

### 6.3 Caption one image (helper function)

In [ ]:
def caption_image(image, model, processor, prompt, max_new_tokens=300):
    messages = [{"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": prompt},
    ]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(model.device, dtype=torch.float16)
    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    text = processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    # Model sometimes wraps JSON in prose or code fences despite instructions — extract the {...} block.
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1:
        try:
            return json.loads(text[start:end + 1]), text
        except json.JSONDecodeError:
            pass
    return None, text  # caption failed to parse — kept as raw text, worth a mis-seeing dossier entry

### 6.4 Caption the corpus once, cache to Drive *(mandatory caching — resumable)*
Skips images already captioned in the cache file, so a disconnect mid-run costs nothing — just re-run this cell.

In [ ]:
CAPTIONS_PATH = CACHE_DIR / "captions.json"

def load_captions():
    if CAPTIONS_PATH.exists():
        with open(CAPTIONS_PATH) as f:
            return json.load(f)
    return {}

captions = load_captions()
already_done = set(captions.keys())
todo = [p for p in image_paths if str(p) not in already_done]
print(f"{len(already_done)} already captioned, {len(todo)} remaining.")

for p in tqdm(todo, desc="Captioning"):
    img = Image.open(p).convert("RGB")
    parsed, raw = caption_image(img, vlm_model, vlm_processor, CAPTION_PROMPT, max_new_tokens=400)
    captions[str(p)] = {"parsed": parsed, "raw": raw, "captioned_at": datetime.now().isoformat()}
    #a disconnect loses at most one image.
    with open(CAPTIONS_PATH, "w") as f:
        json.dump(captions, f, indent=2)

print(f"Done. {len(captions)} total captions cached at {CAPTIONS_PATH}")

### 6.5 Embed captions for retrieval

Captions are prose, often longer than SigLIP2's 64-token text limit, and this is text-to-text similarity (query vs. caption) rather than text-to-image, a dedicated sentence embedding model fits better than reusing the SigLIP2 text tower here. `all-MiniLM-L6-v2` is small enough to run on CPU.

In [ ]:
!pip install -q -U sentence-transformers

from sentence_transformers import SentenceTransformer

sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

def caption_to_text(entry):
    p = entry.get("parsed")
    if not p:
        return entry.get("raw", "")
    return " ".join(str(p.get(f, "")) for f in CAPTION_SCHEMA_FIELDS if p.get(f))

CAPTION_EMB_PATH = CACHE_DIR / "caption_embeddings.npy"
CAPTION_EMB_INDEX_PATH = CACHE_DIR / "caption_embeddings_index.json"

caption_paths = [p for p in image_paths if str(p) in captions]
caption_texts = [caption_to_text(captions[str(p)]) for p in caption_paths]

if CAPTION_EMB_PATH.exists() and CAPTION_EMB_INDEX_PATH.exists():
    with open(CAPTION_EMB_INDEX_PATH) as f:
        idx = json.load(f)
    if idx.get("paths") == [str(p) for p in caption_paths]:
        caption_embeddings = np.load(CAPTION_EMB_PATH)
        print("Loaded cached caption embeddings.")
    else:
        caption_embeddings = sentence_model.encode(caption_texts, normalize_embeddings=True, show_progress_bar=True)
        np.save(CAPTION_EMB_PATH, caption_embeddings)
        json.dump({"paths": [str(p) for p in caption_paths]}, open(CAPTION_EMB_INDEX_PATH, "w"))
else:
    caption_embeddings = sentence_model.encode(caption_texts, normalize_embeddings=True, show_progress_bar=True)
    np.save(CAPTION_EMB_PATH, caption_embeddings)
    json.dump({"paths": [str(p) for p in caption_paths]}, open(CAPTION_EMB_INDEX_PATH, "w"))

print("caption_embeddings shape:", caption_embeddings.shape)

### 6.6 Caption-route retrieval + grounded chat generation


In [ ]:
import textwrap
import matplotlib.pyplot as plt
import matplotlib

def search_captions(query, top_k=None):
    top_k = top_k or CFG["top_k_default"]
    q_emb = sentence_model.encode([query], normalize_embeddings=True)[0]
    sims = caption_embeddings @ q_emb
    order = np.argsort(-sims)[:top_k]
    return [(caption_paths[i], float(sims[i])) for i in order]

def ask_archive(query, top_k=5, max_new_tokens=300):
    results = search_captions(query, top_k=top_k)
    context = "\n".join(
        f"- {p.name}: {caption_to_text(captions[str(p)])}" for p, _ in results
    )
    prompt = (
        f"You are answering questions about a personal image archive using only the descriptions below. "
        f"Be honest if the descriptions don't support a confident answer.\n\n"
        f"Descriptions:\n{context}\n\nQuestion: {query}\nAnswer:"
    )
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(vlm_model.device, dtype=torch.float16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    # Visual UI Update
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 4.5))
    if n == 1:
        axes = [axes]

    for ax, (p, sim) in zip(axes, results):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(f"{p.name}\nsim={sim:.3f}", fontsize=8)
        ax.axis("off")

    # Wrap and print the answer at the top of the plot
    wrapped_answer = "\n".join(textwrap.wrap(f"AI Answer: {answer}", width=100))
    plt.suptitle(f"Question: {query}\n\n{wrapped_answer}",
                 fontsize=10, fontweight='bold', y=1.05, backgroundcolor='#f0f0f0')

    plt.tight_layout()
    plt.show()
    return answer, results

_ = ask_archive("what are the most natural looking images in the collection")

### 6.7 Side-by-side comparison: CLIP/SigLIP2 route vs. caption route
Run the same query through both and log where they agree or diverge; this feeds directly into the question of "when does joint-embedding beat caption-mediated retrieval" question. Run on at least 5 shared queries.

In [ ]:
def compare_routes(query, top_k=5):
    """Run one query through the SigLIP2 route and the caption route and show both top-k rows."""
    clip_res = search(query, top_k=top_k)
    cap_res = search_captions(query, top_k=top_k)
    clip_names = {p.name for p, _ in clip_res}
    cap_names = {p.name for p, _ in cap_res}
    overlap = clip_names & cap_names
    fig, axes = plt.subplots(2, top_k, figsize=(3 * top_k, 6.8))
    for row, (label, res) in enumerate([("SigLIP2", clip_res), ("Caption", cap_res)]):
        for col, (p, sim) in enumerate(res):
            ax = axes[row][col]
            ax.imshow(Image.open(p).convert("RGB"))
            shared = " *" if p.name in overlap else ""
            ax.set_title(f"{label}: {p.name}{shared}\nsim={sim:.3f}", fontsize=8)
            ax.axis("off")
    plt.suptitle(f'Query: "{query}"   ({len(overlap)}/{top_k} shared in top-{top_k}; * = in both)')
    plt.tight_layout()
    plt.show()
    return {"query": query, "shared": sorted(overlap)}


In [ ]:
for q in ["asphalt", "lightning", "fabric or woven texture", "tree", "a galaxy or nebula"]:
    compare_routes(q, top_k=5)

In [ ]:
interesting_queries = [
    "a calm and peaceful scene",
    "a single isolated object with empty space around it",
    "something that looks dangerous or threatening",
    "a perfectly symmetrical pattern",
]

for q in interesting_queries:
    compare_routes(q, top_k=5)

### 6.8 Mis-seeing dossier logger


In [ ]:
# --- Mis-seeing dossier: logger (Strictly Text-Only) ---
DOSSIER_PATH = CACHE_DIR / "mis_seeing_dossier.json"

def _load_dossier():
    if DOSSIER_PATH.exists():
        try:
            with open(DOSSIER_PATH, "r") as f:
                content = f.read().strip()
                if not content:
                    return []
                return json.loads(content)
        except (json.JSONDecodeError, ValueError) as e:
            print(f"⚠️ Dossier file corrupted ({e}). Resetting to empty list to fix the crash.")
            return []
    return []

def log_mis_seeing(query, issue_type, description, model_said, actually_is):
    """
    Logs a text-only entry to the dossier linked to the triggering query.
    """
    dossier = _load_dossier()
    idx = len(dossier) + 1

    dossier.append({
        "query": query,
        "issue_type": issue_type,
        "description": description,
        "model_said": model_said,
        "actually_is": actually_is,
        "logged_at": datetime.now().isoformat(),
    })

    with open(DOSSIER_PATH, "w") as f:
        json.dump(dossier, f, indent=2)
    print(f"Logged #{idx} — Query: '{query}' ({issue_type})")
    return dossier

def export_dossier_markdown(path=None):
    dossier = _load_dossier()
    if not dossier:
        print("No dossier entries yet.")
        return
    path = path or (CACHE_DIR / "mis_seeing_dossier.md")
    lines = ["# Mis-Seeing Dossier — Level 2 (Text-Only)\n"]
    for i, e in enumerate(dossier, 1):
        q_text = e.get('query', e.get('file', 'Unknown Query'))
        lines.append(f"## Entry {i}: Query — \"{q_text}\"\n")
        lines.append(f"**Issue Type:** {e.get('issue_type', 'N/A')}\n")
        lines.append(f"**Model said:** {e.get('model_said', 'N/A')}\n")
        lines.append(f"**Actually is:** {e.get('actually_is', 'N/A')}\n")
        lines.append(f"**Analysis:** {e.get('description', 'N/A')}\n")
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print(f"Exported {len(dossier)} entries -> {path}")

In [ ]:

log_mis_seeing(
    query="asphalt",
    issue_type="retrieval mismatch",
    description="Caption-mediated retrieval for 'asphalt' surfaced this butterfly render at rank 5, "
                "unrelated to any of the genuinely granular dark textures found elsewhere. Likely "
                "matched on incidental shared caption vocabulary rather than visual similarity, since "
                "nothing about a pastel butterfly resembles asphalt. Worth noting the chat model itself "
                "stayed honest and refused to confirm any image as asphalt when asked directly.",
    model_said="chat answer: 'I cannot confirm that any of these images depict asphalt'",
    actually_is="a pastel-colored butterfly render, no textural or color relation to asphalt",
)

log_mis_seeing(
    query="tree",
    issue_type="missed resemblance",
    description="SigLIP2 correctly retrieved this as a tree purely from visual silhouette (conical, "
                "tapering, evergreen-like branch structure). The caption route missed it entirely: "
                "the captioning model described it as abstract geometric/particle structure with no "
                "noted resemblance, so no caption text existed for the query 'tree' to match against. "
                "My schema deliberately discourages forcing resemblances to avoid false positives "
                "elsewhere, which cost recall here on an obvious real case.",
    model_said="captioned as abstract, no clear resemblance noted; chat explicitly denied any tree resemblance",
    actually_is="a clearly tree-shaped particle render, conical evergreen silhouette",
)

log_mis_seeing(
    query="a galaxy or nebula",
    issue_type="rationalized false resemblance",
    description="For 'a galaxy or nebula,' caption-mediated retrieval surfaced this GLSL "
                "shader, and rather than rejecting the match the model stretched to justify it, citing "
                "'radiating streaks,' 'luminous,' and 'glow' as vague evidence of a cosmic resemblance. "
                "These are words shared with genuine nebula-like renders in the corpus, but a symmetrical "
                "mandala bears no real visual resemblance to a galaxy. Different failure mode than the "
                "tree case: here the model overreaches instead of underclaiming.",
    model_said="'radiating streaks and dense fields... somewhat similar to how light and gas flow in nebulae'",
    actually_is="a symmetrical geometric kaleidoscope shader, not a cosmic or celestial form",
)

log_mis_seeing(
    query="a calm and peaceful scene",
    issue_type="mislabeled mood",
    description="This high-contrast, aggressive pattern was captioned with the mood 'serene, "
                "contemplative' which doesn't match what's visually there. "
                "Caption-mediated retrieval trusted that mood label for the query 'a calm and peaceful "
                "scene' and ranked this image first, ahead of genuinely calmer sunset renders elsewhere "
                "in the corpus, and also surfaced two visually chaotic fire-toned images for the same "
                "reason. Shows the caption route inherits whatever mood-reading errors the VLM made "
                "at captioning time, with no way to correct for it at retrieval time.",
    model_said="'serene, contemplative, slightly melancholic, with a hint of warmth'",
    actually_is="a stark, high-contrast symmetrical pattern with no calm but melancholic visual qualities",
)

export_dossier_markdown()

### 6.9 Gradio search interface

In [ ]:
import gradio as gr

def gradio_compare_fn(query, top_k):
    if not query.strip():
        return "Please enter a question.", [], []

    # Level 1 Retrieval (SigLIP2)
    l1_results = search(query, top_k=int(top_k))
    l1_gallery = [(str(p), f"{p.name} | sim={sim:.3f}") for p, sim in l1_results]

    # Level 2 Retrieval & Answer (Captions)
    answer, l2_results = ask_archive(query, top_k=int(top_k))
    l2_gallery = [(str(p), f"{p.name} | sim={sim:.3f}") for p, sim in l2_results]

    return answer, l1_gallery, l2_gallery

with gr.Blocks(title="Seeing Machines — Comparative Atlas") as comparative_demo:
    gr.Markdown("## Seeing Machines — Route Comparison (Level 1 vs Level 2)")
    gr.Markdown("Compare **Direct Retrieval** (SigLIP 2) with **Caption-Mediated Retrieval** (Gemma 3 RAG).")

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Shared Query",
                placeholder="e.g., 'minimalist geometric suns'",
                lines=2
            )
            with gr.Row():
                topk_slider = gr.Slider(1, 10, value=5, step=1, label="Top k")
                compare_btn = gr.Button("Run Comparison", variant="primary")

        with gr.Column(scale=3):
            answer_output = gr.Markdown(label="Level 2 Synthesis", value="*Results will appear here...*")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### Level 1: SigLIP 2 (Visual Embeddings)")
            l1_gallery_output = gr.Gallery(label="L1 Results", columns=3, height="auto")

        with gr.Column():
            gr.Markdown("### Level 2: Captions (Text Search)")
            l2_gallery_output = gr.Gallery(label="L2 Results", columns=3, height="auto")

    # Event wiring
    compare_btn.click(fn=gradio_compare_fn, inputs=[query_input, topk_slider], outputs=[answer_output, l1_gallery_output, l2_gallery_output])
    query_input.submit(fn=gradio_compare_fn, inputs=[query_input, topk_slider], outputs=[answer_output, l1_gallery_output, l2_gallery_output])

comparative_demo.launch(debug=False, share=True)

# 7. Level 3 — The Critic

### 7.1 Hybrid retrieval: fuse SigLIP2 + caption similarity

Design note: SigLIP2 image-text cosine similarity and the sentence-embedding caption similarity live on different scales/distributions, combining raw scores would silently let one route dominate. Min-max normalizing each route's scores *across the full corpus* before fusing puts them on comparable footing; `alpha` then controls the actual tradeoff (alpha=1 → pure SigLIP2, alpha=0 → pure caption route).

In [ ]:
CFG["hybrid_alpha_default"] = 0.5  # 0 = caption-only, 1 = clip-only, 0.5 = even fusion

clip_path_index = {str(p): i for i, p in enumerate(image_paths)}
caption_path_index = {str(p): i for i, p in enumerate(caption_paths)}
common_paths = [p for p in image_paths if str(p) in caption_path_index]

if len(common_paths) < len(image_paths):
    print(f"Note: {len(image_paths) - len(common_paths)} images have no caption yet — "
          f"hybrid retrieval only covers the {len(common_paths)} that do.")

def _minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-8)

def hybrid_search(query, top_k=None, alpha=None):
    top_k = top_k or CFG["top_k_default"]
    alpha = CFG["hybrid_alpha_default"] if alpha is None else alpha

    clip_q = embed_text_query(query, embed_model, embed_processor, device, CFG["text_max_length"])
    cap_q = sentence_model.encode([query], normalize_embeddings=True)[0]

    clip_sims_n = _minmax(image_embeddings @ clip_q)
    cap_sims_n = _minmax(caption_embeddings @ cap_q)

    scored = []
    for p in common_paths:
        ci, ki = clip_path_index[str(p)], caption_path_index[str(p)]
        score = alpha * clip_sims_n[ci] + (1 - alpha) * cap_sims_n[ki]
        scored.append((p, float(score)))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]

def show_hybrid_results(query, top_k=6, alpha=None):
    results = hybrid_search(query, top_k=top_k, alpha=alpha)
    used_alpha = CFG["hybrid_alpha_default"] if alpha is None else alpha
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, (p, s) in zip(axes, results):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(f"{p.name}\nscore={s:.3f}", fontsize=8)
        ax.axis("off")
    plt.suptitle(f'Hybrid (alpha={used_alpha}): "{query}"')
    plt.tight_layout()
    plt.show()
    return results

_ = show_hybrid_results("river", alpha=0.5)

### 7.2 Adjustable-weight Gradio panel
Exposes `alpha` directly in the interface.

In [ ]:
import gradio as gr

def gradio_hybrid_fn(query, top_k, alpha):
    if not query.strip():
        return []
    results = hybrid_search(query, top_k=int(top_k), alpha=alpha)
    return [(str(p), f"{p.name} | score={s:.3f}") for p, s in results]

with gr.Blocks(title="Seeing Machines — Level 3: Hybrid Retrieval") as hybrid_demo:
    gr.Markdown("## Hybrid Retrieval — SigLIP2 ↔ Captions")
    with gr.Row():
        h_query = gr.Textbox(label="Query", scale=3)
        h_topk = gr.Slider(1, 20, value=CFG["top_k_default"], step=1, label="Top k", scale=1)
    h_alpha = gr.Slider(0.0, 1.0, value=CFG["hybrid_alpha_default"], step=0.05,
                         label="alpha (0 = caption-only, 1 = SigLIP2-only)")
    h_btn = gr.Button("Search")
    h_gallery = gr.Gallery(label="Results", columns=4, height="auto")
    h_btn.click(fn=gradio_hybrid_fn, inputs=[h_query, h_topk, h_alpha], outputs=h_gallery)
    h_query.submit(fn=gradio_hybrid_fn, inputs=[h_query, h_topk, h_alpha], outputs=h_gallery)

hybrid_demo.launch(debug=False)

### 7.3 True multimodal grounding — images in the VLM context

Passes the actual retrieved images into the model's context instead of answering from cached text captions. `ask_archive_multimodal` takes the number of images as `top_k` (default 2). The final Gradio demo caps this at 2 images, resized to 448 px, to keep the 4-bit model within T4 memory.

### 7.4 Gold set: candidate inspection and hand labels

The 20 queries below are run through all three routes and inspected visually. I then hand-labelled the correct images per query (1 to 4 each, 41 labels in total) into `GOLD_QUERIES`. Because each query has at most 4 correct images, the best achievable mean precision@5 on this set is 0.41, not 1.0; read the scores in 7.5 against that ceiling.

In [ ]:
gold_candidate_queries = [
    "a human figure", "water or liquid motion", "dense particle cloud",
    "a grid or lattice structure", "warm sunset colors", "cool blue tones",
    "sharp jagged edges", "soft rounded forms", "high motion blur",
    "a single glowing point of light", "dark and shadowy composition",
    "bright overexposed composition", "a spiral or vortex shape",
    "scattered random noise", "a solid continuous surface",
    "layered transparent shapes", "a symmetrical mirrored pattern",
    "an explosion or burst", "thin wispy strands", "a dense solid mass",
]

for q in gold_candidate_queries:
    print(f"\n=========== {q} ===========")
    print("--- CLIP/SigLIP2 ---")
    show_results(q, top_k=6)
    print("--- Caption route ---")
    results = search_captions(q, top_k=6)
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3.2))
    if n == 1: axes = [axes]
    for ax, (p, sim) in zip(axes, results):
        ax.imshow(Image.open(p).convert("RGB")); ax.set_title(f"{p.name}\nsim={sim:.3f}", fontsize=8); ax.axis("off")
    plt.tight_layout(); plt.show()
    print("--- Hybrid (alpha=0.5) ---")
    show_hybrid_results(q, top_k=6, alpha=0.5)

In [ ]:
GOLD_QUERIES = [
    {"query": "a human figure", "correct_files": ["<Screenshot 2026-07-05 200706.png>", "<Screenshot 2026-07-05 195612.png>"]},
    {"query": "water or liquid motion", "correct_files": ["<Screenshot 2026-07-05 200037.png>", "<TDMovieOut.34.jpg>"]},
    {"query": "dense particle cloud", "correct_files": ["<TDMovieOut.188.jpg>", "<TDMovieOut.36.jpg>"]},
    {"query": "a grid or lattice structure", "correct_files": ["<TDMovieOut.139.jpg>", "<TDMovieOut.142.jpg>"]},
    {"query": "warm sunset colors", "correct_files": ["<1 (12).mp4_20250415_012602.575.jpg>", "<TDMovieOut.28.jpg>"]},
    {"query": "cool blue tones", "correct_files": ["<Screenshot 2026-07-05 200037.png>", "<TDMovieOut.63.jpg>", "<TDMovieOut.64.jpg>"]},
    {"query": "sharp jagged edges", "correct_files": ["<TDMovieOut.105.jpg>"]},
    {"query": "soft rounded forms", "correct_files": ["<TDMovieOut.157.jpg>", "<TDMovieOut.103.jpg>"]},
    {"query": "high motion blur", "correct_files": ["<TDMovieOut.156.jpg>", "<TDMovieOut.67.jpg>"]},
    {"query": "a single glowing point of light", "correct_files": ["<TDMovieOut.131.jpg>"]},
    {"query": "dark and shadowy composition", "correct_files": ["<Screenshot 2026-07-05 195716.png>", "<TDMovieOut.31.jpg>"]},
    {"query": "bright overexposed composition", "correct_files": ["<TDMovieOut.113.jpg>", "<TDMovieOut.89.jpg>"]},
    {"query": "a spiral or vortex shape", "correct_files": ["<TDMovieOut.175.jpg>"]},
    {"query": "scattered random noise", "correct_files": ["<TDMovieOut.28 (2).jpg>", "<TDMovieOut.81.jpg>"]},
    {"query": "a solid continuous surface", "correct_files": ["<TDMovieOut.112.jpg>", "<TDMovieOut.174.jpg>"]},
    {"query": "layered transparent shapes", "correct_files": ["<TDMovieOut.17.jpg>", "<TDMovieOut.100.jpg>"]},
    {"query": "a symmetrical mirrored pattern", "correct_files": ["<TDMovieOut.21 (3).jpg>", "<TDMovieOut.23 (2).jpg>", "<TDMovieOut.24 (3).jpg>", "<Screenshot 2026-07-05 201244.png>"]},
    {"query": "an explosion or burst", "correct_files": ["<TDMovieOut.1.jpg>", "<TDMovieOut.2.jpg>", "<TDMovieOut.8.jpg>"]},
    {"query": "thin wispy strands", "correct_files": ["<TDMovieOut.180.jpg>", "<TDMovieOut.59.jpg>"]},
    {"query": "a dense solid mass", "correct_files": ["<TDMovieOut.174.jpg>", "<TDMovieOut.188.jpg>"]},
]
for item in GOLD_QUERIES:
    item["correct_files"] = [f.strip("<>") for f in item["correct_files"]]

# quick sanity check
print(GOLD_QUERIES[14])

### 7.5 Precision@k across all three routes

In [ ]:
def precision_at_k(retrieved_paths, correct_files, k):
    retrieved_names = [Path(p).name for p in retrieved_paths[:k]]
    correct_set = set(correct_files)
    hits = sum(1 for name in retrieved_names if name in correct_set)
    return hits / k

def evaluate_routes(gold_queries, k=5, hybrid_alpha=None):
    rows = []
    for item in tqdm(gold_queries, desc="Evaluating"):
        q, correct = item["query"], item["correct_files"]

        clip_paths = [p for p, _ in search(q, top_k=k)]
        cap_paths = [p for p, _ in search_captions(q, top_k=k)]
        hybrid_paths = [p for p, _ in hybrid_search(q, top_k=k, alpha=hybrid_alpha)]

        rows.append({
            "query": q,
            "clip_only": precision_at_k(clip_paths, correct, k),
            "caption_only": precision_at_k(cap_paths, correct, k),
            "hybrid": precision_at_k(hybrid_paths, correct, k),
        })
    return pd.DataFrame(rows)

In [ ]:
import pandas as pd
eval_df = evaluate_routes(GOLD_QUERIES, k=5)
eval_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Prepare and polish data
avg_scores = eval_df[["clip_only", "caption_only", "hybrid"]].mean().reset_index()
avg_scores.columns = ['Route', 'Score']
route_map = {
    "clip_only": "SigLIP 2 (Direct Visual)",
    "caption_only": "Gemma 3 (Semantic Text)",
    "hybrid": "Hybrid Fusion (Optimized)"
}
avg_scores['Route'] = avg_scores['Route'].map(route_map)
avg_scores = avg_scores.sort_values('Score', ascending=False)

# 2. Set premium aesthetic style
plt.rcParams['font.family'] = 'sans-serif'
sns.set_theme(style="white")
f, ax = plt.subplots(figsize=(12, 7))

# 3. Create a stunning horizontal bar chart with a vibrant palette
palette = sns.color_palette("magma", n_colors=len(avg_scores) + 2).as_hex()[1:4]
bar_plot = sns.barplot(
    x="Score",
    y="Route",
    data=avg_scores,
    hue="Route",
    palette=palette,
    edgecolor="#2c3e50",
    linewidth=2.5,
    legend=False
)

# 4. Add glass-style data labels
for p in ax.patches:
    width = p.get_width()
    ax.annotate(f'{width:.2f}',
                xy=(width, p.get_y() + p.get_height() / 2),
                xytext=(20, 0),
                textcoords="offset points",
                ha='left', va='center',
                fontsize=18, fontweight='900', color='#1a1a1a')

# 5. High-end typography and layout
plt.title("ARCHIVE RETRIEVAL: PERFORMANCE AUDIT", fontsize=26, fontweight='600', pad=45, loc='left', color='#000000')
plt.xlabel("MEAN PRECISION @ K=5", fontsize=14, labelpad=20, fontweight='bold', color='#555555')
plt.ylabel("", fontsize=14)
plt.xlim(0, 1.05)

# 6. Final visual polish
sns.despine(left=True, bottom=True)
ax.grid(axis='x', linestyle=':', alpha=0.5, color='#000000')
ax.tick_params(axis='y', labelsize=15, labelcolor='#333333', length=0)

# Subtle background accent
ax.set_facecolor('#fcfcfc')

plt.tight_layout()
plt.show()

In [ ]:
def ask_archive_multimodal(query, top_k=None, max_new_tokens=300):
    import textwrap
    from PIL import Image
    torch.cuda.empty_cache()

    # Use hybrid search to find the most relevant images
    results = hybrid_search(query, top_k=top_k or 2)

    # Construct multimodal message with actual images
    content = [{"type": "image", "image": Image.open(p).convert("RGB")} for p, _ in results]
    content.append({"type": "text", "text":
        f"Answer the question using only what you can see in these {len(results)} images. "
        f"Be honest if the images don't support a confident answer.\n\nQuestion: {query}"})

    messages = [{"role": "user", "content": content}]

    inputs = vlm_processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt"
    ).to(vlm_model.device, dtype=torch.float16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    answer = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    # Visualization
    n = len(results)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 5))
    if n == 1: axes = [axes]
    for ax, (p, s) in zip(axes, results):
        ax.imshow(Image.open(p).convert("RGB"))
        ax.set_title(f"{p.name}\nScore: {s:.3f}", fontsize=9)
        ax.axis("off")

    wrapped_text = "\n".join(textwrap.wrap(f"VLM Analysis: {answer}", width=70 * n))
    plt.suptitle(f"Query: {query}\n\n{wrapped_text}",
                 fontsize=10, y=1.05, va='bottom', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    plt.tight_layout()
    plt.show()

    return answer, results

# Now run the comparison
for q in ["a spiral or vortex shape", "an explosion or burst"]:
    print(f"### {q}")
    print("TEXT-ONLY (Level 2 — captions in context):")
    ask_archive(q, top_k=3)
    print("IMAGES-IN-CONTEXT (Level 3 — actual pixels in context):")
    ask_archive_multimodal(q, top_k=3)

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")

In [ ]:
import gradio as gr
from PIL import Image
import torch
import gc

def ui_l1_search(query, top_k):
    if not query.strip(): return []
    results = search(query, top_k=int(top_k))
    return [(str(p), f"{p.name} | sim={s:.3f}") for p, s in results]

def ui_l2_chat(query, top_k):
    if not query.strip(): return "", []
    results = search_captions(query, top_k=int(top_k))
    context = "\n".join(f"- {p.name}: {caption_to_text(captions[str(p)])}" for p, _ in results)
    prompt = (f"You are answering questions about a personal image archive using only the descriptions "
              f"below. Be honest if the descriptions don't support a confident answer.\n\n"
              f"Descriptions:\n{context}\n\nQuestion: {query}\nAnswer:")
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    inputs = vlm_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                                return_dict=True, return_tensors="pt").to(vlm_model.device, dtype=torch.float16)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=300, do_sample=False)
    answer = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()
    return answer, [(str(p), f"{p.name} | sim={s:.3f}") for p, s in results]

def ui_l3_hybrid(query, top_k, alpha):
    if not query.strip(): return []
    results = hybrid_search(query, top_k=int(top_k), alpha=alpha)
    return [(str(p), f"{p.name} | score={s:.3f}") for p, s in results]

def ui_l3_multimodal(query):
    if not query.strip(): return "", []

    # 1. Clear memory before starting
    gc.collect()
    torch.cuda.empty_cache()

    # 2. Strict image limit (max 2 images for T4 stability)
    results = hybrid_search(query, top_k=2)

    # 3. Resize images to VLM's native size (448) to save massive VRAM
    images = []
    for p, _ in results:
        img = Image.open(p).convert("RGB")
        img.thumbnail((448, 448))
        images.append(img)

    content = [{"type": "image", "image": img} for img in images]
    content.append({"type": "text", "text": f"Answer using only these {len(images)} images. Be honest.\n\nQuestion: {query}"})

    messages = [{"role": "user", "content": content}]
    inputs = vlm_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                                return_dict=True, return_tensors="pt").to(vlm_model.device, dtype=torch.float16)

    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        out = vlm_model.generate(**inputs, max_new_tokens=300, do_sample=False)

    answer = vlm_processor.decode(out[0][input_len:], skip_special_tokens=True).strip()

    # 4. Cleanup after generation
    del inputs, out
    torch.cuda.empty_cache()

    return answer, [(str(p), p.name) for p, _ in results]

with gr.Blocks(title="Seeing Machines — Full Demo") as master_demo:
    gr.Markdown("# Seeing Machines\nLevel 1, 2 & 3 — pick a tab.")

    with gr.Tab("Level 1 — The Finder"):
        with gr.Row():
            l1_query = gr.Textbox(label="Query", scale=3)
            l1_topk = gr.Slider(1, 20, value=8, step=1, label="Top k", scale=1)
        l1_btn = gr.Button("Search")
        l1_gallery = gr.Gallery(label="Results", columns=4, height="auto")
        l1_btn.click(ui_l1_search, [l1_query, l1_topk], l1_gallery)
        l1_query.submit(ui_l1_search, [l1_query, l1_topk], l1_gallery)

    with gr.Tab("Level 2 — The Companion"):
        with gr.Row():
            l2_query = gr.Textbox(label="Ask your archive", scale=3)
            l2_topk = gr.Slider(1, 10, value=5, step=1, label="Context images", scale=1)
        l2_btn = gr.Button("Ask")
        l2_answer = gr.Textbox(label="Answer", lines=4)
        l2_gallery = gr.Gallery(label="Retrieved (caption route)", columns=5, height="auto")
        l2_btn.click(ui_l2_chat, [l2_query, l2_topk], [l2_answer, l2_gallery])
        l2_query.submit(ui_l2_chat, [l2_query, l2_topk], [l2_answer, l2_gallery])

    with gr.Tab("Level 3 — Hybrid Retrieval"):
        with gr.Row():
            l3_query = gr.Textbox(label="Query", scale=3)
            l3_topk = gr.Slider(1, 20, value=8, step=1, label="Top k", scale=1)
        l3_alpha = gr.Slider(0.0, 1.0, value=0.5, step=0.05, label="alpha (0=captions, 1=SigLIP2)")
        l3_btn = gr.Button("Search")
        l3_gallery = gr.Gallery(label="Results", columns=4, height="auto")
        l3_btn.click(ui_l3_hybrid, [l3_query, l3_topk, l3_alpha], l3_gallery)
        l3_query.submit(ui_l3_hybrid, [l3_query, l3_topk, l3_alpha], l3_gallery)

    with gr.Tab("Level 3 — Multimodal Grounding"):
        gr.Markdown("Passes the retrieved *images themselves* (max 2) into the VLM's context. Resized to 448px for stability.")
        mm_query = gr.Textbox(label="Question")
        mm_btn = gr.Button("Ask with images in context")
        mm_answer = gr.Textbox(label="Answer", lines=4)
        mm_gallery = gr.Gallery(label="Images passed to the model", columns=2, height="auto")
        mm_btn.click(ui_l3_multimodal, [mm_query], [mm_answer, mm_gallery])
        mm_query.submit(ui_l3_multimodal, [mm_query], [mm_answer, mm_gallery])

master_demo.launch(debug=False, share=True)

### 7.6 Interpretation

Hybrid wins on average, but the more interesting story is why, not just that it's higher.

Hybrid isn't just splitting the difference between the two routes, it's beating both of them individually on several queries where each route alone was weak:

*   "dark and shadowy composition" — SigLip2 0.2, caption 0.0, hybrid 0.4
*   "scattered random noise" — SigLip2 0.2, caption 0.0, hybrid 0.4
*   "thin wispy strands" — SigLip2 0.0, caption 0.2, hybrid 0.4

In all three, hybrid scores higher than either single route, not just an average of them. That's a real synergy effect worth naming, when SigLip2 and captions each catch a different correct image (rather than the same one), fusing them lets both correct hits surface in the merged top-5, where neither route's own top-5 alone had room for both.

Then looking at the clear single-route wins, these tell when fusion actually costs something:

*	"cool blue tones" — SigLip2 0.6, caption 0.0, hybrid only 0.2. Here SigLip2 was strongly right and caption was actively wrong, and fusing dragged the hybrid score down from SigLip2's own 0.6. That's a case where the caption route's bad matches diluted a route that didn't need help, a real cost of hybrid, not just upside.

*	"a human figure" / "a grid or lattice structure"; caption wins (0.4 vs SigLip2's 0.0), and hybrid matches caption's score, meaning caption correctly dominated the fusion here.

--

1.  hybrid wins on average
2.  it produces real synergy when the two routes catch different correct images, but
3.  it can actively hurt performance when one route is strongly right and the other is wrong, since averaging pulls the strong signal down.



### Multimodal grounding — observations

Compared text-only (caption-mediated) grounding against passing the actual retrieved images into
the VLM's context, on two queries. These runs used three retrieved images; the final demo caps the count at 2 (see 7.3).

**"a spiral or vortex shape"**  both modes reached the same correct, confident conclusion. All
three retrieved images genuinely are spiral/vortex forms, and both the caption-based answer and
the images-in-context answer identified this clearly, citing concentric rings and swirling
patterns. No meaningful difference here, a clean case where retrieval itself was solid, so
grounding mode didn't matter much.



**"an explosion or burst"**  this is the more revealing case. The multimodal route retrieved
three strong explosion-like renders and gave a confident, accurate answer citing the central
bright point and radiating filaments. The text-only route's retrieval pulled in one unrelated
image (a face-like frame) alongside two correct matches, and the model appropriately
hedged rather than confidently misreporting, a good sign for honesty, but a worse answer overall.


**answer quality
tracked retrieval quality more than it tracked grounding mode.** The multimodal route happened to
get three clean retrieved images; the caption route got contaminated by one bad match. This lines
up with the recurring pattern across the mis-seeing dossier, a bad retrieval upstream tends to
matter more for final answer quality than how the model reasons over the images afterward.

## AI usage statement

This project was developed with assistance from AI tools throughout, used as follows:

- **Claude Sonnet 5 Medium (Free tier) (Anthropic) and Gemini 3.5 Flash (Google):** notebook structure and debugging, grammar and text checks, and the GitHub Pages site (HTML/CSS/JS).
- **Decided by me:** the research question, the corpus and how it was curated, the caption schema, the gold-set labels and the interpretation of results. These were reviewed and decided by me directly against my own corpus.